# 03 Text2Cypher Model Comparison with LangGraph

**Date:** 2026-10-15
**Author:** Xincheng (George) Ding
**Branch:** `george`

**Purpose:** First Graph RAG experiment. A small LangGraph pipeline turns a natural-language question into Cypher, guards it (read-only), runs it on Neo4j, and compares the result with the gold answer. The same pipeline is run for every model listed in `.env`, so we get a model-by-model scorecard on the same questions.

**Needs network** for LLM APIs and Neo4j Aura. Set `DRY_RUN = True` to exercise the graph logic with a fake model and no database (useful offline).

**Input:** gold questions below (the 5 queries from notebook 02, extend to 20), `fa26/.env`
**Output:** `fa26/data/processed/03_text2cypher_scorecard_<timestamp>.csv`

**Install once (base env):** `pip install langgraph langchain-core langchain-openai langchain-anthropic neo4j python-dotenv`

Scope note (2026-10-22). The five questions here are development-time diagnostics for the Text2Cypher pipeline, not the team benchmark. Formal scores come from notebook 05 (free Cypher) and 06 (hybrid router) on the 20-question benchmark in fa26/benchmark/. Do not compare numbers across the two.

## 1. Setup

In [1]:
from pathlib import Path
from datetime import datetime
import os, re, time, json
import pandas as pd
from dotenv import load_dotenv

DRY_RUN = False          # True: fake LLM, no Neo4j. False: real models + Aura.

# Locate fa26/ regardless of the kernel's working directory
FA26 = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".env.example").exists() or p.name == "fa26")
PROCESSED = FA26 / "data" / "processed"; PROCESSED.mkdir(parents=True, exist_ok=True)
assert (FA26 / ".env").exists(), f"No .env found at {FA26}"
load_dotenv(FA26 / ".env", override=True)
print("FA26 =", FA26)
RUN_ID = datetime.now().strftime("%Y-%m-%d_%H%M")
print("run", RUN_ID, "| dry run:", DRY_RUN)

FA26 = /Users/xiding/Desktop/IATI/ds-humanitarian-ai-iati/fa26
run 2026-10-22_0437 | dry run: False


## 2. Schema description handed to the LLM

This is the "manual" Text2Cypher relies on. It mirrors schema v0.1 from notebook 02, which was the graph in Aura when this notebook ran (2026-10-15 to 10-22).

*Update 2026-11-17: the team graph is now loaded by Alex's `fa26/retrieval/load_neo4j.py` (schema `retrieval/schema_prototype.py`): funding is `(Organisation)-[:PROVIDES]->(Transaction)`, `Transaction.value` is a number (plus exact `value_cents`), dates are typed, and `type_label`, `value_num` and `hunger_basis` do not exist. The schema text and the gold queries below are kept as run, except the Oxfam America identifier in the schema text, corrected to the value in the data (`US-EIN-23-7069110`); the saved outputs were produced with the old, wrong identifier. This notebook is a record of the October diagnostic; a live rerun against the current graph needs the schema text and the gold queries updated. Notebooks 05 and 06 carry hand-written schema text for the current graph (not Alex's generated `retrieval/schema.xml`).*

In [2]:
SCHEMA = '''
Node labels and properties:
  (:Activity {iati_identifier, title, description, publisher, status, hunger_basis, start_planned, end_planned, default_currency})
  (:Organisation {ref, name, type, type_label})        // type_label e.g. 'Government', 'International NGO', 'Multilateral'
  (:Country {code, name, hotspot_tier})                 // code ISO-2 e.g. 'SD'; hotspot_tier '1' (highest) .. '3'
  (:Sector {sector_key, code, vocabulary, vocabulary_label, name, is_hunger})
  (:Transaction {tx_id, type, type_label, date, value, value_num, currency, value_date})
        // type_label one of: 'Incoming Funds', 'Outgoing Commitment', 'Disbursement', 'Expenditure'
  (:Budget {budget_id, type, status, period_start, period_end, value, currency})
  (:Location {location_id, name, lat, lon})
  (:SDG {sdg_key, code, name})

Relationships:
  (:Organisation)-[:REPORTS]->(:Activity)               // the publisher of the activity
  (:Organisation)-[:PARTICIPATES_IN {role}]->(:Activity)
  (:Activity)-[:TARGETS {percentage, is_hotspot}]->(:Country)
  (:Activity)-[:CLASSIFIED_AS {percentage}]->(:Sector)
  (:Activity)-[:LOCATED_AT]->(:Location)
  (:Activity)-[:CONTRIBUTES_TO]->(:SDG)
  (:Activity)-[:HAS_TRANSACTION]->(:Transaction)
  (:Activity)-[:HAS_BUDGET]->(:Budget)
  (:Transaction)-[:PROVIDED_BY]->(:Organisation)        // funder
  (:Transaction)-[:RECEIVED_BY]->(:Organisation)        // recipient

Publishers (Organisation.ref): Oxfam GB 'GB-CHC-202918', Oxfam America 'US-EIN-23-7069110', WFP 'XM-DAC-41140'.
Money: sum Transaction.value_num and always group by Transaction.currency (no FX conversion exists).
'''

## 3. Gold questions

Each item has a question, a gold Cypher (from notebook 02) and a short note on what counts as correct. Start with 5, grow to 20 with the client's real questions.

In [3]:
GOLD = [
    {"id": "Q1", "question": "How many activities does each publisher run in each hotspot country?",
     "cypher": '''MATCH (o:Organisation)-[:REPORTS]->(a:Activity)-[:TARGETS]->(c:Country)
RETURN o.name AS publisher, c.name AS country, count(DISTINCT a) AS activities ORDER BY publisher, country'''},
    {"id": "Q2", "question": "What is the total disbursement amount reported by each publisher, broken down by currency?",
     "cypher": '''MATCH (o:Organisation)-[:REPORTS]->(a:Activity)-[:HAS_TRANSACTION]->(t:Transaction)
WHERE t.type_label = 'Disbursement'
RETURN o.name AS publisher, t.currency AS currency, round(sum(t.value_num)) AS total ORDER BY publisher, currency'''},
    {"id": "Q3", "question": "Who are the top 10 funders of Oxfam GB's activities by incoming funds?",
     "cypher": '''MATCH (:Organisation {ref:'GB-CHC-202918'})-[:REPORTS]->(a:Activity)-[:HAS_TRANSACTION]->(t:Transaction)-[:PROVIDED_BY]->(p:Organisation)
WHERE t.type_label = 'Incoming Funds' AND t.value_num > 0
RETURN p.name AS provider, t.currency AS currency, round(sum(t.value_num)) AS total ORDER BY total DESC LIMIT 10'''},
    {"id": "Q4", "question": "For each publisher, how many activities are hunger-related because of a sector code versus a keyword match?",
     "cypher": '''MATCH (o:Organisation)-[:REPORTS]->(a:Activity)
RETURN o.name AS publisher, a.hunger_basis AS basis, count(a) AS activities ORDER BY publisher, basis'''},
    {"id": "Q5", "question": "How many organisations have no IATI identifier, grouped by organisation type?",
     "cypher": '''MATCH (p:Organisation) WHERE p.ref STARTS WITH 'name:'
RETURN p.type_label AS org_type, count(p) AS orgs ORDER BY orgs DESC'''},
]
len(GOLD)

5

## 4. Model registry (from `.env`)

Every entry is `(name, provider, model)`. OpenAI-compatible providers share one client class and differ only by `base_url`. Entries whose key is missing are skipped automatically.

In [4]:
def env(k, default=None): return os.environ.get(k, default) or default

MODELS = []
for m in (env("ANTHROPIC_EVAL_MODELS", "") or "").split(","):
    if m.strip() and env("ANTHROPIC_API_KEY"):
        MODELS.append({"name": f"anthropic/{m.strip()}", "provider": "anthropic", "model": m.strip()})
for prov, key, url, model in [
    ("openai",   "OPENAI_API_KEY",    "OPENAI_BASE_URL",    "OPENAI_MODEL"),
    ("groq",     "GROQ_API_KEY",      "GROQ_BASE_URL",      "GROQ_MODEL"),
    ("nvidia",   "NVIDIA_API_KEY",    "NVIDIA_BASE_URL",    "NVIDIA_MODEL"),
    ("zai",      "ZAI_API_KEY",       "ZAI_BASE_URL",       "ZAI_MODEL"),
    ("dashscope","DASHSCOPE_API_KEY", "DASHSCOPE_BASE_URL", "DASHSCOPE_MODEL"),
    ("deepseek", "DEEPSEEK_API_KEY",  "DEEPSEEK_BASE_URL",  "DEEPSEEK_MODEL"),
]:
    if env(key):
        # every env var named <PROV>_MODEL or <PROV>_MODEL_<SUFFIX> registers one model, e.g. ZAI_MODEL and ZAI_MODEL_FLASH
        for k, v in sorted(os.environ.items()):
            if (k == model or k.startswith(model + "_")) and v.strip():
                MODELS.append({"name": f"{prov}/{v.strip()}", "provider": "openai_compat",
                               "model": v.strip(), "api_key": env(key), "base_url": env(url)})
if DRY_RUN:
    MODELS = [{"name": "fake/echo-gold", "provider": "fake"}, {"name": "fake/bad-cypher", "provider": "fake_bad"}]
assert MODELS, "No models registered: check ANTHROPIC_API_KEY / ANTHROPIC_EVAL_MODELS in fa26/.env"
pd.DataFrame(MODELS)[["name", "provider"]]

,name,provider
0,anthropic/claude-opus-4-6,anthropic
1,anthropic/claude-sonnet-4-6,anthropic
2,anthropic/claude-haiku-4-5,anthropic
3,zai/glm-5.3,openai_compat
4,zai/glm-4.7-flash,openai_compat


In [5]:
# Verify Anthropic model IDs against what this API key can actually see; drop the ones that do not exist
if not DRY_RUN and env("ANTHROPIC_API_KEY"):
    import anthropic
    for k in ("ANTHROPIC_BASE_URL", "ANTHROPIC_AUTH_TOKEN", "ANTHROPIC_API_URL", "HTTPS_PROXY", "HTTP_PROXY"):
        if os.environ.get(k): print(f"NOTE: {k} is set in the environment ->", os.environ[k][:60])
    key = os.environ["ANTHROPIC_API_KEY"]
    ANTHROPIC_HEADERS = {"anthropic-workspace-id": env("ANTHROPIC_WORKSPACE_ID")} if env("ANTHROPIC_WORKSPACE_ID") else {}
    print("key prefix:", key[:10], "| length:", len(key))
    try:
        available = {m.id for m in anthropic.Anthropic(api_key=key, base_url="https://api.anthropic.com", default_headers=ANTHROPIC_HEADERS).models.list(limit=100).data}
    except anthropic.APIStatusError as e:
        print("STATUS", e.status_code, "BODY", e.body)
        (FA26 / "data" / "interim" / "anthropic_error.txt").write_text(
            f"status={e.status_code}\nbody={e.body}\nkey_prefix={key[:10]} len={len(key)}\n"
            + "\n".join(f"{k}={os.environ.get(k)}" for k in ("ANTHROPIC_BASE_URL","ANTHROPIC_AUTH_TOKEN","HTTPS_PROXY")))
        raise
    print("Available Claude models:", sorted(available))
    keep, dropped = [], []
    for spec in MODELS:
        if spec["provider"] == "anthropic" and spec["model"] not in available:
            dropped.append(spec["model"])
        else:
            keep.append(spec)
    if dropped:
        print("Dropped (not available to this key):", dropped, "-> fix ANTHROPIC_EVAL_MODELS in .env")
    MODELS = keep
    assert MODELS, "No usable models left"
pd.DataFrame(MODELS)[["name", "provider"]]

key prefix: sk-ant-api | length: 108
Available Claude models: ['claude-fable-5', 'claude-fable-5-1', 'claude-haiku-4-5-20251001', 'claude-haiku-5-5', 'claude-opus-4-5-20251101', 'claude-opus-4-6', 'claude-opus-4-7', 'claude-opus-4-8', 'claude-opus-5', 'claude-opus-5-5', 'claude-sonnet-4-5-20250929', 'claude-sonnet-4-6', 'claude-sonnet-5', 'claude-sonnet-5-5']
Dropped (not available to this key): ['claude-haiku-4-5'] -> fix ANTHROPIC_EVAL_MODELS in .env


,name,provider
0,anthropic/claude-opus-4-6,anthropic
1,anthropic/claude-sonnet-4-6,anthropic
2,zai/glm-5.3,openai_compat
3,zai/glm-4.7-flash,openai_compat


In [6]:
def make_llm(spec):
    if spec["provider"] == "anthropic":
        from langchain_anthropic import ChatAnthropic
        hdr = {"anthropic-workspace-id": env("ANTHROPIC_WORKSPACE_ID")} if env("ANTHROPIC_WORKSPACE_ID") else None
        return ChatAnthropic(model=spec["model"], temperature=0, max_tokens=1024, default_headers=hdr)
    if spec["provider"] == "openai_compat":
        from langchain_openai import ChatOpenAI
        return ChatOpenAI(model=spec["model"], api_key=spec["api_key"], base_url=spec["base_url"], temperature=0)
    if spec["provider"] in ("fake", "fake_bad"):
        from langchain_core.language_models.fake_chat_models import FakeListChatModel
        if spec["provider"] == "fake":
            return FakeListChatModel(responses=["```cypher\n" + g["cypher"] + "\n```" for g in GOLD])
        return FakeListChatModel(responses=["MATCH (n) DETACH DELETE n", "MATCH (a:Activity) RETURN a.publisher, count(a"] * len(GOLD))
    raise ValueError(spec)

## 5. The LangGraph pipeline

```
question ─► generate_cypher ─► guard ─► execute ─► compare ─► END
                 ▲               │ (unsafe / syntax)
                 └───── retry ◄──┘  (max 1 retry with the error message)
```

State is a plain dict. Each node returns the keys it changes.

In [7]:
from typing import TypedDict, Optional
from langgraph.graph import StateGraph, END
from langchain_core.messages import SystemMessage, HumanMessage

class S(TypedDict, total=False):
    qid: str; question: str; gold_cypher: str
    cypher: str; error: Optional[str]; retries: int
    rows: list; gold_rows: list; match: Optional[bool]; match_kind: Optional[str]; latency_s: float; raw: str

FORBIDDEN = re.compile(r"\b(CREATE|MERGE|DELETE|DETACH|SET|REMOVE|DROP|CALL\s+db\.|LOAD\s+CSV|apoc\.)\b", re.I)

def extract_cypher(text):
    m = re.search(r"```(?:cypher)?\s*(.*?)```", text, re.S)
    return (m.group(1) if m else text).strip().rstrip(";")

def build_graph(llm, run_cypher):
    def generate(st):
        sys_msg = SystemMessage(content="You translate questions about humanitarian aid data into a single read-only Neo4j Cypher query. "
                                        "Use only the labels, properties and relationships in the schema. Return only the query in a ```cypher block.\n\nSCHEMA:\n" + SCHEMA)
        human = st["question"] if not st.get("error") else f"{st['question']}\n\nYour previous query failed: {st['error']}\nPrevious query:\n{st['cypher']}\nFix it."
        t0 = time.time()
        out = llm.invoke([sys_msg, HumanMessage(content=human)])
        return {"raw": out.content, "cypher": extract_cypher(out.content), "latency_s": time.time() - t0, "error": None}

    def guard(st):
        if FORBIDDEN.search(st["cypher"]): return {"error": "unsafe: write or procedure call detected"}
        if not re.match(r"^\s*(MATCH|OPTIONAL MATCH|WITH|UNWIND|RETURN)", st["cypher"], re.I): return {"error": "not a read query"}
        return {"error": None}

    def execute(st):
        try:
            return {"rows": run_cypher(st["cypher"]), "error": None}
        except Exception as e:
            return {"rows": [], "error": f"execution error: {str(e)[:300]}"}

    def compare(st):
        # Semantic comparison: ignore column names and column order, compare the VALUES.
        # exact  = same rows (as multisets of normalised values)
        # subset = same row count and every gold row's values appear inside the model's row (model added extra columns)
        # An empty gold result is never a pass (guards against an empty database).
        def nv(v):
            if isinstance(v, bool): return str(v)
            if isinstance(v, (int, float)): return round(float(v), 0)
            return str(v).strip().lower()
        def rowset(rows): return sorted(tuple(sorted(map(str, (nv(v) for v in r.values())))) for r in rows)
        gold = run_cypher(st["gold_cypher"])
        rows = st.get("rows") or []
        if not gold or not rows:
            return {"gold_rows": gold, "match": False, "match_kind": "empty"}
        if rowset(rows) == rowset(gold):
            return {"gold_rows": gold, "match": True, "match_kind": "exact"}
        gs, ms = rowset(gold), rowset(rows)
        subset = len(gs) == len(ms) and all(any(set(g) <= set(m) for m in ms) for g in gs)
        return {"gold_rows": gold, "match": subset, "match_kind": "subset" if subset else "different"}

    def after_guard(st):  return "retry" if st.get("error") and st.get("retries", 0) < 1 else ("execute" if not st.get("error") else "compare")
    def after_exec(st):   return "retry" if st.get("error") and st.get("retries", 0) < 1 else "compare"
    def bump(st):         return {"retries": st.get("retries", 0) + 1}

    g = StateGraph(S)
    g.add_node("generate_cypher", generate); g.add_node("guard", guard); g.add_node("execute", execute)
    g.add_node("compare", compare); g.add_node("retry", bump)
    g.set_entry_point("generate_cypher")
    g.add_edge("generate_cypher", "guard")
    g.add_conditional_edges("guard", after_guard, {"retry": "retry", "execute": "execute", "compare": "compare"})
    g.add_conditional_edges("execute", after_exec, {"retry": "retry", "compare": "compare"})
    g.add_edge("retry", "generate_cypher"); g.add_edge("compare", END)
    return g.compile()

## 6. Cypher executor (Neo4j Aura, or a stub in dry run)

In [8]:
if DRY_RUN:
    _stub = {g["cypher"]: [{"ok": g["id"]}] for g in GOLD}
    def run_cypher(q):
        if q.strip().endswith("count(a"): raise ValueError("Invalid input: expected ')'")
        return _stub.get(q.strip().rstrip(";"), [{"rows": "unknown-query"}])
else:
    from neo4j import GraphDatabase
    driver = GraphDatabase.driver(os.environ["NEO4J_URI"], auth=(os.environ["NEO4J_USERNAME"], os.environ["NEO4J_PASSWORD"]))
    driver.verify_connectivity()
    def run_cypher(q):
        with driver.session(database=env("NEO4J_DATABASE", "neo4j")) as s:
            return s.run(q).data()
    print("Neo4j connected")

Neo4j connected


## 7. Run every model on every question

In [9]:
records = []
for spec in MODELS:
    app = build_graph(make_llm(spec), run_cypher)
    for g in GOLD:
        try:
            st = app.invoke({"qid": g["id"], "question": g["question"], "gold_cypher": g["cypher"], "retries": 0})
        except Exception as e:
            st = {"error": f"{type(e).__name__}: {str(e)[:200]}", "retries": 0}
        records.append({"run": RUN_ID, "model": spec["name"], "qid": g["id"], "match": st.get("match"), "match_kind": st.get("match_kind"),
                        "error": st.get("error"), "retries": st.get("retries", 0),
                        "latency_s": round(st.get("latency_s", 0), 2), "cypher": st.get("cypher")})
        print(f"{spec['name']:<32} {g['id']}  match={st.get('match')} ({st.get('match_kind')})  retries={st.get('retries',0)}  err={st.get('error')}")
results = pd.DataFrame(records)

anthropic/claude-opus-4-6        Q1  match=True (subset)  retries=0  err=None
anthropic/claude-opus-4-6        Q2  match=True (subset)  retries=0  err=None
anthropic/claude-opus-4-6        Q3  match=False (empty)  retries=0  err=None
anthropic/claude-opus-4-6        Q4  match=False (different)  retries=0  err=None
anthropic/claude-opus-4-6        Q5  match=False (empty)  retries=0  err=None
anthropic/claude-sonnet-4-6      Q1  match=True (subset)  retries=0  err=None
anthropic/claude-sonnet-4-6      Q2  match=True (subset)  retries=0  err=None
anthropic/claude-sonnet-4-6      Q3  match=False (different)  retries=0  err=None
anthropic/claude-sonnet-4-6      Q4  match=False (different)  retries=0  err=None
anthropic/claude-sonnet-4-6      Q5  match=False (empty)  retries=0  err=None
zai/glm-5.3                      Q1  match=True (subset)  retries=0  err=None
zai/glm-5.3                      Q2  match=True (exact)  retries=0  err=None
zai/glm-5.3                      Q3  match=False (dif

## 8. Scorecard

In [10]:
scorecard = (results.assign(match=results["match"].fillna(False).astype(bool))
             .groupby("model").agg(accuracy=("match", "mean"), avg_latency_s=("latency_s", "mean"),
                                   retries=("retries", "sum"), errors=("error", lambda s: s.notna().sum()))
             .round(2).sort_values("accuracy", ascending=False))
out = PROCESSED / f"03_text2cypher_scorecard_{RUN_ID}.csv"
results.to_csv(out, index=False)
print("Saved", out)
scorecard

Saved /Users/xiding/Desktop/IATI/ds-humanitarian-ai-iati/fa26/data/processed/03_text2cypher_scorecard_2026-10-22_0437.csv


,accuracy,avg_latency_s,retries,errors
model,,,,
anthropic/claude-opus-4-6,0.4,3.68,0,0
anthropic/claude-sonnet-4-6,0.4,3.51,0,0
zai/glm-5.3,0.4,16.21,0,0
zai/glm-4.7-flash,0.0,8.59,0,3


## 9. How to read the results and what comes next

- How `match` is decided: in the first version (2026-10-15) a result matched only if its rows were identical to the gold rows, column names included, so a query with the right numbers under different column names counted as wrong. Since 2026-10-22 (commit f7a9080, "semantic compare in 03") `compare` in section 5 ignores column names and order, rounds numbers to whole units, lower-cases strings, and also accepts a subset match (same row count, every gold row's values contained in a result row). The saved outputs use the second version.
- A model that fails the `guard` is trying to write to the graph. Keep the guard even in production.
- This notebook is a diagnostic on 5 hand-written questions, not the team benchmark. Formal scores come from Deniz's benchmark in notebooks 05 and 06.